# Response of atmosphere to sea surface temperature bump using jax.jvp

The coupled model comes through `jem.configurations.load` -- the recipe door onto the shipped, validated `aquaplanet-slab` configuration (issue #131) -- rather than composing Hydra by hand: what this notebook demonstrates (differentiating through a coupled run with `jax.jvp`) has nothing to do with how the model was assembled.

In [ ]:
from pathlib import Path

import jax
import jax.numpy as jnp

from jem import configurations, plot, read_field, replace_field

output_dir = (Path("output") / "01-03_sst_response_jvp").resolve()
output_dir.mkdir(parents=True, exist_ok=True)

## Build the shipped model

In [ ]:
# `seaice="none"` is the door's equivalent of the CLI's `seaice=none`:
# a two-component (atmosphere/ocean) model, matching what this notebook
# always coupled -- see `jem/configurations.py` (issue #131).
exp = configurations.load("aquaplanet-slab", seaice="none")
coupler = exp.coupler
coupler

## Taking Gradient of the Coupled Model

Mathematically speaking, we are trying to assess
$$
x_{\mathrm{final}} = F_{t_0, T}(x_0)
$$
where $x_0$ and $x_{\mathrm{final}}$ the initial and final states of the system, $t_0$ the initial time, $T$ the length of simulation time, and $F_{t_0, T}$ the trajectory function that maps the input state from time $t = t_0$ to $t = t_0 +  T$.

The following code attempts to compute the sensitivity of the trajectory function to a pulse function $g(x)$. That is, let initial condition $x_0$ be perturbed as
$$
    x'_0(x; \epsilon) = x_0(x) + \epsilon g(x)
$$
the response of the final state to $\delta(x)$ will then be
$$
x'_{\mathrm{final}} = x_{\mathrm{final}} + \epsilon \, \delta x_{\mathrm{final}}
$$
through which the sensitivity of final statet to $g$, noted as $s_{x_{\mathrm{final}}}$, is formally defined as
$$
s_{x_{\mathrm{final}}} = \frac{\partial x'_{\mathrm{final}} }{\partial \epsilon}
$$

The entire computation is achieved using `jax.jvp`. 

In [ ]:
trajectory = coupler.generate_trajectory_function(5)
initial_carry = coupler.initialize()


@jax.jit
def forecast(sst):
    """Run five coupled days from an ocean initialised with `sst`.

    The carry is rebuilt here rather than mutated because `sst` is
    the argument being differentiated, so it has to reach the
    trajectory function through the carry it is called with.
    """
    carry = replace_field(initial_carry, "ocn.state.sea_surface_temperature", sst)
    final, _ = trajectory(carry)
    return (read_field(final, "ocn.state.sea_surface_temperature"),
            read_field(final, "atm.derived.v0"))

In [ ]:
sst_initial = read_field(initial_carry, "ocn.state.sea_surface_temperature")

# Put a point SST perturbation in the middle of the domain.
shape2d = sst_initial.shape
tangent = jnp.zeros_like(sst_initial).at[shape2d[0] // 2, shape2d[1] // 2].set(1.0)

# jax.jvp gives the sensitivity of SST and surface meridional wind
# to that point perturbation.
(sst_final, v_final), (d_sst, d_v) = jax.jvp(forecast, (sst_initial,), (tangent,))

## Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

# All four fields share the ocean's grid (aquaplanet: same grid as
# the atmosphere), so one pair of degree coordinates covers them all.
grid = coupler.components["ocn"].grid
lon = xr.DataArray(np.degrees(grid.longitude_axis_radian), dims="lon")
lat = xr.DataArray(np.degrees(grid.latitude_axis_radian), dims="lat")


def as_field(array):
    return xr.DataArray(array, dims=("lon", "lat"), coords={"lon": lon, "lat": lat})


fig, ax = plt.subplots(3, 2, figsize=(12, 16))

sst_levels = jnp.linspace(-2, 35, 11)
tangent_sst_levels = jnp.linspace(-1, 1, 11) * 0.5
v_levels = jnp.linspace(-1, 1, 11) * 5
tangent_v_levels = jnp.linspace(-1, 1, 11) * 0.2

plot.map_plot(as_field(sst_initial - 273.15), ax=ax[0, 0], levels=sst_levels,
              title="(a) $\\mathrm{SST}_\\mathrm{init}$")
plot.map_plot(as_field(tangent), ax=ax[0, 1], levels=tangent_sst_levels, cmap="bwr",
              title="(b) $\\partial \\mathrm{SST}_\\mathrm{init}$")
plot.map_plot(as_field(sst_final - 273.15), ax=ax[1, 0], levels=sst_levels,
              title="(c) $\\mathrm{SST}_\\mathrm{final}$")
plot.map_plot(as_field(d_sst), ax=ax[1, 1], levels=tangent_sst_levels, cmap="bwr",
              title="(d) $\\partial \\mathrm{SST}_\\mathrm{final}$")
plot.map_plot(as_field(v_final), ax=ax[2, 0], levels=v_levels, cmap="bwr",
              title="(e) $\\mathrm{v}_\\mathrm{final}$")
plot.map_plot(as_field(d_v), ax=ax[2, 1], levels=tangent_v_levels, cmap="bwr",
              title="(f) $\\partial \\mathrm{v}_\\mathrm{final}$")

fig.suptitle("Response time: 5.0 days")
for _ax in ax.flatten():
    _ax.set_xlabel("longitude [deg]")
    _ax.set_ylabel("latitude [deg]")
plt.tight_layout()

In [ ]:
fig.savefig(output_dir / "sensitivity.png", dpi=150)